# Paper Figure 6 — selected-motif support and robustness

Upstream analyses:
- `revision_06_archetype_interpretation_K46_independent_groups`
- `revision_07_K46_edge_trim_heldout_top5_sensitivity`

This notebook is deliberately presentation-only: it loads the participant-
level and edge-trim outputs from those analysis notebooks and assembles the
reviewer-facing robustness figure.

Panels:
- **A:** participant-level spatial support across all 108 independent participants
- **B:** leave-one-out spatial-map similarity by condition
- **C:** condition-specific support magnitude
- **D:** held-out top-5 accuracy before and after edge trimming
- **E:** change in selection frequency after edge trimming for the five locked motifs

This layout preserves the important nuance that the overall sparse phenomenon
survives edge trimming while A5 is comparatively boundary-sensitive.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ARCHETYPES = [4, 5, 6, 12, 8]
CONDITIONS = ["intact", "word", "rest"]
COLORS = {"intact":"#7B3294", "word":"#2CA25F", "rest":"#555555"}

SUPPORT_DIR = Path("revision_05_archetype_interpretation_and_support")
EDGE_DIR = Path("msaa_edge_trim_sensitivity_K46_top5")

SUPPORT_CSV = SUPPORT_DIR / "selected_archetype_support_by_independent_participant.csv"
MAP_CSV = SUPPORT_DIR / "selected_archetype_map_similarity_by_condition.csv"
COND_CSV = SUPPORT_DIR / "selected_archetype_condition_summary_independent.csv"

EDGE_ACC_CSV = EDGE_DIR / "edge_trim_accuracy_summary.csv"
EDGE_FREQ_CSV = EDGE_DIR / "edge_trim_selection_frequency_comparison.csv"

OUTPUT_DIR = Path("paper_figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_PNG = OUTPUT_DIR / "Figure6_selected_motif_support_robustness.png"
FIG_PDF = OUTPUT_DIR / "Figure6_selected_motif_support_robustness.pdf"

for p in [SUPPORT_CSV, MAP_CSV, COND_CSV, EDGE_ACC_CSV, EDGE_FREQ_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing required input: {p}\n"
            "Run the corresponding upstream analysis notebook first."
        )

support = pd.read_csv(SUPPORT_CSV)
map_summary = pd.read_csv(MAP_CSV)
cond_summary = pd.read_csv(COND_CSV)
edge_acc = pd.read_csv(EDGE_ACC_CSV)
edge_freq = pd.read_csv(EDGE_FREQ_CSV)

display(edge_acc)

In [ ]:
# Derived summaries for plotting
def sem_from_sd(sd, n):
    return np.asarray(sd, dtype=float) / np.sqrt(np.asarray(n, dtype=float))

map_summary["sem_loo_cosine"] = sem_from_sd(
    map_summary["sd_loo_cosine"],
    map_summary["n"],
)

# Ensure selected order
support["archetype"] = pd.Categorical(
    support["archetype"],
    categories=ARCHETYPES,
    ordered=True,
)
map_summary["archetype"] = pd.Categorical(
    map_summary["archetype"],
    categories=ARCHETYPES,
    ordered=True,
)
cond_summary["archetype"] = pd.Categorical(
    cond_summary["archetype"],
    categories=ARCHETYPES,
    ordered=True,
)

# Edge-frequency change for locked motifs only
ef = edge_freq[edge_freq["archetype"].isin(ARCHETYPES)].copy()
ef["delta_frequency"] = ef["trim_frequency"] - ef["baseline_frequency"]
delta_heat = (
    ef.pivot_table(
        index="archetype",
        columns="condition",
        values="delta_frequency",
        aggfunc="mean",
    )
    .reindex(index=ARCHETYPES, columns=CONDITIONS)
)

support.to_csv(
    OUTPUT_DIR / "Figure6_support_source.csv",
    index=False,
)
edge_acc.to_csv(
    OUTPUT_DIR / "Figure6_edge_trim_source.csv",
    index=False,
)

In [ ]:
fig = plt.figure(figsize=(15.5, 10.5))
gs = fig.add_gridspec(
    2, 3,
    height_ratios=[1, 1],
    width_ratios=[1.15, 1, 1],
    hspace=0.35,
    wspace=0.30,
)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[0, 2])
axD = fig.add_subplot(gs[1, 0:2])
axE = fig.add_subplot(gs[1, 2])

# ------------------------------------------------------------
# A — participant support
# ------------------------------------------------------------
rng = np.random.default_rng(42)
positions = np.arange(len(ARCHETYPES))
box_data = []

for pos, k in enumerate(ARCHETYPES):
    vals = support.loc[
        support["archetype"].astype(int) == k,
        "support_l2",
    ].to_numpy(dtype=float)
    box_data.append(vals)
    jitter = rng.normal(0, 0.045, size=len(vals))
    axA.scatter(
        np.full(len(vals), pos) + jitter,
        vals,
        s=13,
        alpha=0.25,
    )

axA.boxplot(
    box_data,
    positions=positions,
    widths=0.45,
    showfliers=False,
)
axA.set_xticks(positions)
axA.set_xticklabels([f"A{k}" for k in ARCHETYPES])
axA.set_ylabel(r"Spatial support $\|S_{k,:}\|_2$")
axA.set_title("Broad participant support", fontweight="bold")

# ------------------------------------------------------------
# B — leave-one-out map similarity
# ------------------------------------------------------------
x = np.arange(len(ARCHETYPES))
offsets = {"intact":-0.18, "word":0.0, "rest":0.18}

for condition in CONDITIONS:
    cur = (
        map_summary[map_summary["condition"] == condition]
        .set_index("archetype")
        .reindex(ARCHETYPES)
    )
    axB.errorbar(
        x + offsets[condition],
        cur["mean_loo_cosine"],
        yerr=cur["sem_loo_cosine"],
        marker="o",
        linewidth=1.5,
        capsize=2,
        color=COLORS[condition],
        label=condition,
    )

axB.set_xticks(x)
axB.set_xticklabels([f"A{k}" for k in ARCHETYPES])
axB.set_ylabel("Leave-one-out map cosine similarity")
axB.set_ylim(0, 1)
axB.set_title("Spatial-map reproducibility", fontweight="bold")
axB.legend(frameon=False, fontsize=9)

# ------------------------------------------------------------
# C — support magnitude by condition
# ------------------------------------------------------------
for condition in CONDITIONS:
    cur = (
        cond_summary[cond_summary["condition"] == condition]
        .set_index("archetype")
        .reindex(ARCHETYPES)
    )
    axC.errorbar(
        x + offsets[condition],
        cur["mean_support"],
        yerr=cur["sem_support"],
        marker="o",
        linewidth=1.5,
        capsize=2,
        color=COLORS[condition],
        label=condition,
    )

axC.set_xticks(x)
axC.set_xticklabels([f"A{k}" for k in ARCHETYPES])
axC.set_ylabel(r"Mean spatial support $\|S_{k,:}\|_2$")
axC.set_title("Condition-specific deployment", fontweight="bold")

# ------------------------------------------------------------
# D — edge-trim accuracy
# ------------------------------------------------------------
ea = edge_acc.set_index("condition").reindex(CONDITIONS)
xx = np.arange(len(CONDITIONS))

axD.errorbar(
    xx - 0.08,
    ea["baseline_accuracy"],
    yerr=ea["baseline_accuracy_sem"],
    marker="o",
    capsize=3,
    linewidth=1.8,
    label="Untrimmed",
)
axD.errorbar(
    xx + 0.08,
    ea["trim_accuracy"],
    yerr=ea["trim_accuracy_sem"],
    marker="o",
    capsize=3,
    linewidth=1.8,
    label="Trim first/last 10 TRs",
)

for i, condition in enumerate(CONDITIONS):
    frac = ea.loc[condition, "fraction_preserved"]
    if np.isfinite(frac) and condition != "rest":
        axD.text(
            i,
            max(ea.loc[condition, "baseline_accuracy"],
                ea.loc[condition, "trim_accuracy"]) * 1.07,
            f"{100*frac:.0f}% above-chance retained",
            ha="center",
            va="bottom",
            fontsize=9,
        )

axD.set_xticks(xx)
axD.set_xticklabels(["Intact", "Word", "Rest"])
axD.set_ylabel("Held-out top-5 accuracy")
axD.set_title("Sparse decoding remains after edge exclusion", fontweight="bold")
axD.legend(frameon=False, fontsize=9)

# ------------------------------------------------------------
# E — edge-trim change in selected motifs
# ------------------------------------------------------------
heat = delta_heat.to_numpy(dtype=float)
v = np.nanmax(np.abs(heat))
im = axE.imshow(
    heat,
    aspect="auto",
    vmin=-v,
    vmax=v,
    cmap="coolwarm",
)

axE.set_xticks(np.arange(len(CONDITIONS)))
axE.set_xticklabels(["Intact", "Word", "Rest"], rotation=30, ha="right")
axE.set_yticks(np.arange(len(ARCHETYPES)))
axE.set_yticklabels([f"A{k}" for k in ARCHETYPES])
axE.set_title("Change in top-5 selection frequency", fontweight="bold")

for r in range(heat.shape[0]):
    for c in range(heat.shape[1]):
        val = heat[r, c]
        axE.text(c, r, f"{val:+.2f}", ha="center", va="center", fontsize=9)

cb = fig.colorbar(im, ax=axE, fraction=0.046, pad=0.04)
cb.set_label("Trimmed − untrimmed")

# panel labels
for ax, letter in zip([axA, axB, axC, axD, axE], list("ABCDE")):
    ax.text(
        -0.12, 1.04, letter,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
    )
    ax.tick_params(labelsize=10)

fig.suptitle(
    "Selected archetypes are broadly supported and robust",
    fontsize=16,
    fontweight="bold",
    y=0.99,
)

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)

### Figure 6 interpretation

The broad-support panels address whether the selected motifs are driven by a
small number of participants. The edge-trim panels address whether sparse
decoding is primarily a run-boundary artifact.

The intended interpretation is:

> The overall sparse top-5 phenomenon is robust to removal of run-boundary
> timepoints, while A5 is comparatively boundary-sensitive.

Do not replace A5 post hoc.